In [1]:
import pandas as pd
df = pd.read_csv("/content/Diseases_Symptoms.csv")
df.head()

,Name,Symptoms,Treatments,Disease_Code,Contagious,Chronic
0,Gestational Cholestasis,"Itchy skin, particularly on the hands and feet",NaN,D001,False,False
1,Injury to Internal Organ,"Abdominal pain, bleeding, organ dysfunction","Immediate medical attention, diagnostic tests,...",D002,False,False
2,Scabies,"Intense itching, especially at night, small bl...",Prescription medications (topical or oral scab...,D003,False,True
3,Congenital Glaucoma,"Cloudy or hazy eyes, excessive tearing, sensit...","Surgery (e.g., trabeculotomy, goniotomy) to cr...",D004,False,True
4,Avoidant/Restrictive Food Intake Disorder (ARFID),Avoidance or restriction of certain foods or e...,"Nutritional counseling, psychotherapy (such as...",D005,False,True


In [3]:
# Check for missing values
print("Missing values before handling:")
print(df.isnull().sum())

# Fill missing values in 'Treatments' with a placeholder
df['Treatments'] = df['Treatments'].fillna('No specific treatment mentioned')

print("\nMissing values after handling:")
print(df.isnull().sum())

Missing values before handling:
Name            0
Symptoms        0
Treatments      0
Disease_Code    0
Contagious      0
Chronic         0
dtype: int64

Missing values after handling:
Name            0
Symptoms        0
Treatments      0
Disease_Code    0
Contagious      0
Chronic         0
dtype: int64


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the dataset
df = pd.read_csv("/content/Diseases_Symptoms.csv")

# --- TF-IDF for the 'Symptoms' column ---
tfidf_symptom = TfidfVectorizer(stop_words='english')
symptom_tfidf = tfidf_symptom.fit_transform(df['Symptoms'].astype(str))

# Convert to DataFrame
symptom_tfidf_df = pd.DataFrame(
symptom_tfidf.toarray(),
columns=tfidf_symptom.get_feature_names_out()
 )

print("TF-IDF for Symptoms column:")
print(symptom_tfidf_df.head())

# --- TF-IDF for the 'Treatments' column ---
tfidf_treatment = TfidfVectorizer(stop_words='english')
treatment_tfidf = tfidf_treatment.fit_transform(df['Treatments'].astype(str))

# Convert to DataFrame
treatment_tfidf_df = pd.DataFrame(
treatment_tfidf.toarray(),
columns=tfidf_treatment.get_feature_names_out()
 )

print("\nTF-IDF for Treatments column:")
print(treatment_tfidf_df.head())

# --- Optional: Save results ---
symptom_tfidf_df.to_csv("tfidf_symptom.csv", index=False)
treatment_tfidf_df.to_csv("tfidf_treatment.csv", index=False)

TF-IDF for Symptoms column:
    24  abdomen  abdominal  abnormal  abnormalities  abrasions  abscess  \
0  0.0      0.0   0.000000       0.0            0.0        0.0      0.0   
1  0.0      0.0   0.348812       0.0            0.0        0.0      0.0   
2  0.0      0.0   0.000000       0.0            0.0        0.0      0.0   
3  0.0      0.0   0.000000       0.0            0.0        0.0      0.0   
4  0.0      0.0   0.000000       0.0            0.0        0.0      0.0   

   absence  absent  aches  ...  worthlessness  wound  wounds  xanthomas  year  \
0      0.0     0.0    0.0  ...            0.0    0.0     0.0        0.0   0.0   
1      0.0     0.0    0.0  ...            0.0    0.0     0.0        0.0   0.0   
2      0.0     0.0    0.0  ...            0.0    0.0     0.0        0.0   0.0   
3      0.0     0.0    0.0  ...            0.0    0.0     0.0        0.0   0.0   
4      0.0     0.0    0.0  ...            0.0    0.0     0.0        0.0   0.0   

   years  yellow  yellowing  yello

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("/content/Diseases_Symptoms.csv")

# Combine symptom and treatment text (optional but helpful)
df['text'] = df['Symptoms'].astype(str) + " " + df['Treatments'].astype(str)

# Define features (X) and target (y)
X = df['text']
y = df['Name']  # Use 'Name' as the target column for disease prediction

# Split data into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )

# Convert text to TF-IDF features
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Example prediction
example = ["fever cough runny nose"]  # replace with any new symptom text
example_tfidf = tfidf.transform(example)
predicted_disease = model.predict(example_tfidf)
print("\nPredicted Disease:", predicted_disease[0])

✅ Accuracy: 0.012345679012345678

Classification Report:
                                                  precision    recall  f1-score   support

                                        Abscess       0.00      0.00      0.00         1
                             Acute Bronchospasm       0.00      0.00      0.00         1
                            Acute Kidney Injury       0.00      0.00      0.00         1
                                Acute Sinusitis       0.00      0.00      0.00         0
                                 Adrenal Cancer       0.00      0.00      0.00         1
                       Anemia due to Malignancy       0.00      0.00      0.00         1
                      Anemia of Chronic Disease       0.00      0.00      0.00         1
                                        Anxiety       0.00      0.00      0.00         1
                                      Arthritis       0.00      0.00      0.00         1
                                         Asthma    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='macro'))
print("Recall:", recall_score(y_test, y_pred, average='macro'))
print("F1 Score:", f1_score(y_test, y_pred, average='macro'))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.012345679012345678
Precision: 0.0012077294685990338
Recall: 0.010869565217391304
F1 Score: 0.002173913043478261

Confusion Matrix:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Classification Report:

                                                 precision    recall  f1-score   support

                                        Abscess       0.00      0.00      0.00         1
                             Acute Bronchospasm       0.00      0.00      0.00         1
                            Acute Kidney Injury       0.00      0.00      0.00         1
                                Acute Sinusitis       0.00      0.00      0.00         0
                                 Adrenal Cancer       0.00      0.00      0.00         1
                       Anemia due to Malignancy       0.00      0.00      0.00         1
                      Anemia of Chronic Disease       0.00      0.00      0.00         

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

# **word2vec with Lstm and Bi-Lstm**

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
elmo = hub.load("https://tfhub.dev/google/elmo/3")

In [7]:
import sys

!{sys.executable} -m pip install gensim tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.3 MB/s eta 0:00:00


In [8]:
!pip install gensim
# ============================================
# 1. IMPORTS
# ============================================
from gensim.models import Word2Vec
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split


# ============================================
# 2. LOAD YOUR DATA
# ============================================
df = pd.read_csv("/content/Diseases_Symptoms.csv")  # modify filename

# Combine text fields
df["Text"] = df["Name"].astype(str) + " " + df["Symptoms"].astype(str) + " " + df["Treatments"].astype(str)

# Target label → Disease_Code
labels = df["Disease_Code"]


# ============================================
# 3. TOKENIZE TEXT FOR WORD2VEC
# ============================================
df["Tokens"] = df["Text"].apply(lambda x: x.lower().split())
sentences = df["Tokens"].tolist()

# Train Word2Vec
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)


# ============================================
# 4. TRAIN / TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(df["Tokens"], labels, test_size=0.2, random_state=42)

X_train = X_train.tolist()
X_test = X_test.tolist()


# ============================================
# 5. TOKENIZER → SEQUENCES
# ============================================
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 100
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(labels) # Fit on all possible labels
y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)


# ============================================
# 6. CREATE EMBEDDING MATRIX
# ============================================
word_index = tokenizer.word_index
embedding_dim = 100
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))

for word, i in word_index.items():
    if word in w2v.wv:
        embedding_matrix[i] = w2v.wv[word]


# ============================================
# 7. LSTM MODEL
# ============================================
model_lstm = Sequential()
model_lstm.add(Embedding(input_dim=len(word_index)+1,
                          output_dim=embedding_dim,
                          weights=[embedding_matrix],
                          trainable=False))

model_lstm.add(LSTM(128))
model_lstm.add(Dropout(0.3))
model_lstm.add(Dense(64, activation='relu'))
model_lstm.add(Dense(num_classes, activation='softmax'))

model_lstm.compile(loss='sparse_categorical_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

print("\n🚀 Training LSTM...")
model_lstm.fit(X_train_pad, y_train_encoded, epochs=5, batch_size=32, validation_split=0.1)

# Predictions
y_pred_lstm_encoded = model_lstm.predict(X_test_pad).argmax(axis=1)

# Get unique labels in the test set and their corresponding names for the classification report
unique_labels_in_test_lstm = np.unique(y_test_encoded)
target_names_for_report_lstm = label_encoder.inverse_transform(unique_labels_in_test_lstm)

print("\n========== LSTM RESULTS ==========")
print("Accuracy:", accuracy_score(y_test_encoded, y_pred_lstm_encoded))
print(classification_report(y_test_encoded, y_pred_lstm_encoded,
                            labels=unique_labels_in_test_lstm,
                            target_names=target_names_for_report_lstm))


# ============================================
# 8. BI-LSTM MODEL
# ============================================
model_bilstm = Sequential()
model_bilstm.add(Embedding(input_dim=len(word_index)+1,
                           output_dim=embedding_dim,
                           weights=[embedding_matrix],
                           trainable=False))

model_bilstm.add(Bidirectional(LSTM(128)))
model_bilstm.add(Dropout(0.3))
model_bilstm.add(Dense(64, activation='relu'))
model_bilstm.add(Dense(num_classes, activation='softmax'))

model_bilstm.compile(loss='sparse_categorical_crossentropy',
                    optimizer='adam',
                    metrics=['accuracy'])

print("\n🚀 Training Bi-LSTM...")
model_bilstm.fit(X_train_pad, y_train_encoded, epochs=5, batch_size=32, validation_split=0.1)

# Predictions
y_pred_bilstm_encoded = model_bilstm.predict(X_test_pad).argmax(axis=1)

# Get unique labels in the test set and their corresponding names for the classification report
unique_labels_in_test_bilstm = np.unique(y_test_encoded)
target_names_for_report_bilstm = label_encoder.inverse_transform(unique_labels_in_test_bilstm)

print("\n========== BI-LSTM RESULTS ==========")
print("Accuracy:", accuracy_score(y_test_encoded, y_pred_bilstm_encoded))
print(classification_report(y_test_encoded, y_pred_bilstm_encoded,
                            labels=unique_labels_in_test_bilstm,
                            target_names=target_names_for_report_bilstm))


🚀 Training LSTM...
Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 79ms/step - accuracy: 0.0000e+00 - loss: 5.9799 - val_accuracy: 0.0000e+00 - val_loss: 5.9836
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.0000e+00 - loss: 5.9759 - val_accuracy: 0.0000e+00 - val_loss: 5.9930
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0058 - loss: 5.9711 - val_accuracy: 0.0000e+00 - val_loss: 6.0166
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0000e+00 - loss: 5.9419 - val_accuracy: 0.0000e+00 - val_loss: 6.5056
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0000e+00 - loss: 5.8494 - val_accuracy: 0.0000e+00 - val_loss: 6.6742
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

========== LSTM RESULTS ==========
Accuracy: 0.0
              precision    recall  f1-score   support

        D001       0.00      0.00      0.00       1.0
        D006       0.00      0.00      0.00       1.0
        D010       0.00      0.00      0.00       1.0
       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 100ms/step - accuracy: 0.0000e+00 - loss: 5.9803 - val_accuracy: 0.0000e+00 - val_loss: 5.9863
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0027 - loss: 5.9739 - val_accuracy: 0.0000e+00 - val_loss: 6.0001
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 9.4046e-04 - loss: 5.9643 - val_accuracy: 0.0000e+00 - val_loss: 6.0459
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 9.4046e-04 - loss: 5.9073 - val_accuracy: 0.0000e+00 - val_loss: 6.6699
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.0017 - loss: 5.7638 - val_accuracy: 0.0000e+00 - val_loss: 6.9260
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 257ms/step

========== BI-LSTM RESULTS ==========
Accuracy: 0.012345679012345678
              precision    recall  f1-score   support

        D001       0.00      0.00      0.00         1
        D006       0.00      0.00      0.00         1
        D010       0.00      0.00      0.00         1
        D016       0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# **Bert with Lstm and Bi-Lstm**

In [9]:
# ==========================================
# 0️⃣ INSTALL LIBRARIES
# ==========================================
!pip install torch transformers numpy pandas scikit-learn nltk tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import numpy as np
import nltk
from tqdm import tqdm

nltk.download('punkt')

# ==========================================
# 1️⃣ LOAD DATA
# ==========================================
df = pd.read_csv("/content/Diseases_Symptoms.csv")

# Use Symptoms as input and Name as output
df = df[['Symptoms', 'Name']]
df.dropna(subset=['Symptoms', 'Name'], inplace=True)

# Encode labels
label_to_id = {label: idx for idx, label in enumerate(df['Name'].unique())}
id_to_label = {idx: label for label, idx in label_to_id.items()}  # For evaluation
df['label'] = df['Name'].map(label_to_id)

# ==========================================
# 2️⃣ TRAIN–TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    df['Symptoms'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# ==========================================
# 3️⃣ TEXT DATASET
# ==========================================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=MAX_LEN,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# ==========================================
# 4️⃣ BERT + BiLSTM MODEL
# ==========================================
class BERT_LSTM(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=len(label_to_id)):
        super(BERT_LSTM, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():  # Freeze BERT for faster training
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        lstm_out, _ = self.lstm(last_hidden_state)
        pooled = torch.mean(lstm_out, dim=1)
        output = self.fc(self.dropout(pooled))
        return output

# ==========================================
# 5️⃣ TRAINING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERT_LSTM().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

# ==========================================
# 6️⃣ EVALUATION
# ==========================================
model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        outputs = model(input_ids, attention_mask)
        preds_batch = torch.argmax(outputs, dim=1).cpu().numpy()

        preds.extend(preds_batch)
        true_labels.extend(labels)

# Accuracy
print("Accuracy:", accuracy_score(true_labels, preds))

# Get unique labels in test set
unique_labels = sorted(list(set(true_labels)))

# Map them to names
target_names = [id_to_label[i] for i in unique_labels]

# Classification report
print(classification_report(true_labels, preds, labels=unique_labels, target_names=target_names))



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Epoch 2/3: 100%|██████████| 41/41 [00:03<00:00, 12.62it/s]


Epoch 2 Loss: 5.9572


Epoch 3/3: 100%|██████████| 41/41 [00:02<00:00, 16.63it/s]


Epoch 3 Loss: 5.9172
Accuracy: 0.0
                                                 precision    recall  f1-score   support

                        Gestational Cholestasis       0.00      0.00      0.00       1.0
                            Subdural hemorrhage       0.00      0.00      0.00       1.0
                                         Myopia       0.00      0.00      0.00       1.0
                                        Anxiety       0.00      0.00      0.00       1.0
                                   Food Allergy       0.00      0.00      0.00       1.0
                          Overflow Incontinence       0.00      0.00      0.00       1.0
                                    Presbycusis       0.00      0.00      0.00       1.0
                            Opioid Use Disorder       0.00      0.00      0.00       1.0
                              Esophageal Cancer       0.00      0.00      0.00       1.0
                              Polycythemia Vera       0.00      0.00      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
